# Week 1, Lab 4 — A minimal ReAct loop

**ReAct** = Reason, Act, Observe, repeat until the model says it is done.

Frameworks hide this loop. You should be able to write it in ~40 lines.


## 1. Setup


In [1]:
WEEK = 'Week 1'
LAB = 'Lab 4 — ReAct loop'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 1 / Lab 4 — ReAct loop
Backend: ollama
Need Ollama running: `ollama serve` and `ollama pull llama3.2:1b`
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [ ]:
# if BACKEND == "huggingface":
#     %pip install -q transformers torch accelerate fastapi uvicorn pydantic
# else:
#     %pip install -q ollama pydantic


## 2. The loop


In [2]:
STOP = "FINAL"
TOOLS = {
    "calculator": calculator,
    "lookup_fact": lookup_fact,
    "today_date": today_date,
}

SYSTEM = """You are a ReAct agent. Each turn, output ONLY JSON as one of:
{"thought": "...", "name": "<tool>", "arguments": {...}}
{"thought": "...", "final": "<answer to the user>"}

Tools: calculator(expression), lookup_fact(topic), today_date()
Never invent tool results. After you have enough observations, set final.
"""

def run_agent(question: str, max_steps: int = 6) -> str:
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    for step in range(1, max_steps + 1):
        reply = local_chat(messages, max_new_tokens=160, temperature=0.1)
        print(f"\n--- step {step} ---\n{reply}")
        obj = extract_json_object(reply) or {}
        if "final" in obj:
            return str(obj["final"])
        name = obj.get("name")
        args = obj.get("arguments") or {}
        if name in TOOLS:
            result = TOOLS[name](**args) if args else TOOLS[name]()
        else:
            result = f"unknown tool {name!r}"
        print("OBS:", result)
        messages.append({"role": "assistant", "content": reply})
        messages.append({"role": "user", "content": f"OBSERVATION: {result}"})
    return "Stopped: max steps reached."

print(run_agent("What is 19*21, and what is LangGraph?"))



--- step 1 ---
{"thought": "19 * 21", "final": "399", "name": "calculator(expression)"}
{"thought": "LangGraph is a graph database that allows for efficient querying of large amounts of data.", "final": "LangGraph"}
OBS: unknown tool None

--- step 2 ---
{"thought": "unknown tool", "arguments": {"tool": "None"}}
OBS: unknown tool None

--- step 3 ---
{"thought": "unknown tool", "final": "None"}
None


## 3. Failure modes to notice

Small models may: skip JSON, call the same tool twice, or 'final' too early. That is expected. Later weeks add retries, graphs, and guardrails because of this.

## 4. Exercise

1. Cap `max_steps` at 2 and describe what is lost.
2. Add a `scratchpad` list in Python (not in the prompt) and print it after the run.
3. Compare this loop to Week 4's LangGraph router — same idea, better structure.

**Next:** `lab5_mini_project.ipynb` — research-and-summarize without a framework.
